# **Notebook 02 — Data Visualisation**

## Objectives

* Explore the dataset through visualisations to understand fraud patterns
* Create 7+ different plot types to answer BR1
* Provide textual interpretation for each visualisation
* Validate project hypotheses with statistical tests

## Inputs

* `data/creditcard.csv`

## Outputs

* Visual insights answering BR1
* Statistical validation of H1, H2, H3
* Identified patterns for feature engineering decisions

---
## Change working directory

In [1]:
import os

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir(os.path.dirname(current_dir))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/dok.stv/Documents/Projects/credit-card-fraud-detection


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, chi2_contingency

df = pd.read_csv("data/creditcard.csv")
print(f"Loaded {df.shape[0]:,} transactions with {df.shape[1]} features")

FileNotFoundError: [Errno 2] No such file or directory: 'data/creditcard.csv'

---
## Plot 1: Class Distribution (Bar Chart)

First thing to visualise is the class balance. This tells us how severe the imbalance problem is.

In [ ]:
class_counts = df['Class'].value_counts()

fig = px.bar(
    x=['Legitimate', 'Fraud'],
    y=class_counts.values,
    color=['Legitimate', 'Fraud'],
    color_discrete_map={'Legitimate': '#636EFA', 'Fraud': '#EF553B'},
    labels={'x': 'Transaction Class', 'y': 'Count'},
    title='Transaction Class Distribution'
)
fig.update_layout(showlegend=False)
fig.show()

### Interpretation

The dataset is extremely imbalanced: 284,315 legitimate transactions (99.83%) vs 492 fraud cases (0.17%). This 577:1 ratio means that a naive model predicting "legitimate" for everything would achieve 99.83% accuracy while catching zero fraud. This confirms the need for specialised techniques like SMOTE oversampling during the modelling phase.

---
## Plot 2: Amount Distribution (Histogram with Box Plot)

Comparing how transaction amounts differ between fraud and legitimate classes.

In [ ]:
fig = px.histogram(
    df, x='Amount', color='Class',
    marginal='box', barmode='overlay',
    color_discrete_map={0: '#636EFA', 1: '#EF553B'},
    opacity=0.7,
    title='Transaction Amount Distribution by Class',
    labels={'Class': 'Transaction Class'}
)
fig.update_layout(xaxis_title='Amount (€)', yaxis_title='Count')
fig.show()

In [ ]:
# Zoom in on fraud amounts specifically
print("Amount statistics:")
print(f"\nLegitimate transactions:")
print(f"  Mean:   €{df[df['Class'] == 0]['Amount'].mean():.2f}")
print(f"  Median: €{df[df['Class'] == 0]['Amount'].median():.2f}")
print(f"  Max:    €{df[df['Class'] == 0]['Amount'].max():.2f}")

print(f"\nFraudulent transactions:")
print(f"  Mean:   €{df[df['Class'] == 1]['Amount'].mean():.2f}")
print(f"  Median: €{df[df['Class'] == 1]['Amount'].median():.2f}")
print(f"  Max:    €{df[df['Class'] == 1]['Amount'].max():.2f}")

### Interpretation

Fraudulent transactions cluster at lower amounts compared to legitimate ones. The median fraud amount is noticeably lower than the median legitimate amount. This suggests fraudsters may deliberately keep amounts low to avoid triggering existing rule-based detection systems. The long tail in legitimate transactions reflects normal high-value purchases that the model must learn to distinguish from fraud.

---
## Plot 3: Feature Correlation Heatmap

Identifying which PCA components correlate most strongly with the fraud class.

In [ ]:
# Find top correlated features with the target
correlations = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)

print("Top 12 features by absolute correlation with fraud:")
print(correlations.head(12))

In [ ]:
# Build heatmap for top 12 features + Class
top_features = correlations.head(12).index.tolist() + ['Class']

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    df[top_features].corr(),
    annot=True, cmap='RdBu_r',
    center=0, fmt='.2f', ax=ax
)
ax.set_title('Correlation Heatmap — Top 12 Features vs Fraud', fontsize=14)
plt.tight_layout()
plt.show()

### Interpretation

V14, V12, and V10 show the strongest negative correlations with the fraud class, while V11, V4, and V2 show notable positive correlations. This means that when V14 has very negative values, a transaction is more likely to be fraudulent. Amount has a weak but present correlation. These relationships will be formally tested in the hypothesis validation section below.

---
## Plot 4: Box Plot — Amount by Class

A clearer view of how amount distributions differ between fraud and legitimate.


In [ ]:
fig = px.box(
    df, x='Class', y='Amount',
    color='Class',
    color_discrete_map={0: '#636EFA', 1: '#EF553B'},
    title='Amount Distribution: Fraud vs Legitimate',
    labels={'Class': 'Transaction Class', 'Amount': 'Amount (€)'}
)
fig.update_layout(showlegend=False)
fig.show()


### Interpretation

The interquartile range for fraud transactions is significantly narrower and lower than for legitimate transactions. Multiple outliers exist in both classes, but legitimate transactions have a much wider spread reflecting normal consumer spending patterns from small daily purchases to large payments.

---
## Plot 5: Violin Plots — Top Discriminating PCA Features

Examining the distribution shape of the most discriminating features to understand how they separate fraud from legitimate.